In [1]:
from efficient_apriori import apriori

watch_history = [
    ("Inception", "Interstellar", "The Dark Knight"),
    ("Inception", "The Dark Knight", "Batman Begins"),
    ("Inception", "Interstellar", "Tenet"),
    ("The Dark Knight", "Joker", "Batman Begins"),
    ("Interstellar", "Gravity", "The Martian"),
    ("The Dark Knight", "Inception", "Joker"),
    ("Interstellar", "Tenet", "Arrival"),
    ("Joker", "The Batman", "The Dark Knight"),
    ("Inception", "Tenet", "Interstellar"),
    ("The Batman", "Joker", "Dark Knight Rises"),
]

In [2]:
itemsets, rules = apriori(watch_history, min_support=0.2, min_confidence=0.5)

print("Total rules:", len(rules))
for r in sorted(rules, key=lambda r: -r.lift):
    print(r, f"[lift={r.lift:.2f}]")

Total rules: 16
{The Batman} -> {Joker} (conf: 1.000, supp: 0.200, lift: 2.500, conv: 600000000.000) [lift=2.50]
{Joker} -> {The Batman} (conf: 0.500, supp: 0.200, lift: 2.500, conv: 1.600) [lift=2.50]
{Inception, Interstellar} -> {Tenet} (conf: 0.667, supp: 0.200, lift: 2.222, conv: 2.100) [lift=2.22]
{Tenet} -> {Inception, Interstellar} (conf: 0.667, supp: 0.200, lift: 2.222, conv: 2.100) [lift=2.22]
{Batman Begins} -> {The Dark Knight} (conf: 1.000, supp: 0.200, lift: 2.000, conv: 500000000.000) [lift=2.00]
{Tenet} -> {Interstellar} (conf: 1.000, supp: 0.300, lift: 2.000, conv: 500000000.000) [lift=2.00]
{Interstellar} -> {Tenet} (conf: 0.600, supp: 0.300, lift: 2.000, conv: 1.750) [lift=2.00]
{Inception, Tenet} -> {Interstellar} (conf: 1.000, supp: 0.200, lift: 2.000, conv: 500000000.000) [lift=2.00]
{The Dark Knight} -> {Joker} (conf: 0.600, supp: 0.300, lift: 1.500, conv: 1.500) [lift=1.50]
{Joker} -> {The Dark Knight} (conf: 0.750, supp: 0.300, lift: 1.500, conv: 2.000) [lift=1.

In [3]:
original_history = list(watch_history)  # keep the baseline for comparison

watch_history = list(original_history)
watch_history[4] = ("Friends", "The Office", "Brooklyn Nine-Nine")
watch_history[6] = ("Friends", "The Office", "Parks and Recreation")


def rules_dict(data, min_support=0.2, min_confidence=0.5):
    _, rs = apriori(data, min_support=min_support, min_confidence=min_confidence)
    return {(tuple(sorted(r.lhs)), tuple(sorted(r.rhs))): r for r in rs}


before = rules_dict(original_history)
after = rules_dict(watch_history)

print("Rules before:", len(before), "| after:", len(after))

print("\nNew rules:")
for k in sorted(set(after) - set(before), key=lambda k: -after[k].lift):
    r = after[k]
    print(f"  {set(k[0])} -> {set(k[1])}  supp={r.support:.2f} conf={r.confidence:.2f} lift={r.lift:.2f}")

print("\nLost rules:", [(set(k[0]), set(k[1])) for k in set(before) - set(after)])

print("\nOld rules that changed (support / lift):")
for k in sorted(set(before) & set(after), key=lambda k: -after[k].lift):
    x, y = before[k], after[k]
    if (x.support, x.lift) != (y.support, y.lift):
        print(f"  {set(k[0])} -> {set(k[1])}  supp {x.support:.2f}->{y.support:.2f}  lift {x.lift:.2f}->{y.lift:.2f}")

Rules before: 16 | after: 19

New rules:
  {'The Office'} -> {'Friends'}  supp=0.20 conf=1.00 lift=5.00
  {'Friends'} -> {'The Office'}  supp=0.20 conf=1.00 lift=5.00
  {'Interstellar'} -> {'Tenet', 'Inception'}  supp=0.20 conf=0.67 lift=3.33

Lost rules: []

Old rules that changed (support / lift):
  {'Tenet', 'Inception'} -> {'Interstellar'}  supp 0.20->0.20  lift 2.00->3.33
  {'Interstellar'} -> {'Tenet'}  supp 0.30->0.20  lift 2.00->3.33
  {'Tenet'} -> {'Interstellar'}  supp 0.30->0.20  lift 2.00->3.33
  {'Tenet'} -> {'Inception', 'Interstellar'}  supp 0.20->0.20  lift 2.22->3.33
  {'Inception', 'Interstellar'} -> {'Tenet'}  supp 0.20->0.20  lift 2.22->3.33
  {'Tenet', 'Interstellar'} -> {'Inception'}  supp 0.20->0.20  lift 1.33->2.00
  {'Inception'} -> {'Interstellar'}  supp 0.30->0.30  lift 1.20->2.00
  {'Interstellar'} -> {'Inception'}  supp 0.30->0.30  lift 1.20->2.00
  {'Tenet'} -> {'Inception'}  supp 0.20->0.20  lift 1.33->2.00


In [4]:
itemsets, rules = apriori(watch_history, min_support=0.2, min_confidence=0.5)

movie = "Joker"  # try "Interstellar" or "Friends" too

matches = [r for r in rules if movie in r.lhs or movie in r.rhs]
for r in sorted(matches, key=lambda r: -r.lift):
    print(r, f"[lift={r.lift:.2f}]")

print("\nRules involving", movie + ":", len(matches))
print("Highest lift:", round(max(r.lift for r in matches), 2))

{The Batman} -> {Joker} (conf: 1.000, supp: 0.200, lift: 2.500, conv: 600000000.000) [lift=2.50]
{Joker} -> {The Batman} (conf: 0.500, supp: 0.200, lift: 2.500, conv: 1.600) [lift=2.50]
{The Dark Knight} -> {Joker} (conf: 0.600, supp: 0.300, lift: 1.500, conv: 1.500) [lift=1.50]
{Joker} -> {The Dark Knight} (conf: 0.750, supp: 0.300, lift: 1.500, conv: 2.000) [lift=1.50]

Rules involving Joker: 4
Highest lift: 2.5


In [5]:
settings = [
    (0.3, 0.6),  # higher (the lab's change)
    (0.2, 0.5),  # original
    (0.1, 0.3),  # lower
]

for min_sup, min_conf in settings:
    itemsets, rules = apriori(
        watch_history,
        min_support=min_sup,
        min_confidence=min_conf,
    )
    lifts = [r.lift for r in rules]
    print(f"min_support={min_sup}, min_confidence={min_conf}")
    print("  Total rules:", len(rules))
    print(f"  Lift min/avg/max: {min(lifts):.2f} / {sum(lifts)/len(lifts):.2f} / {max(lifts):.2f}")

min_support=0.3, min_confidence=0.6
  Total rules: 6
  Lift min/avg/max: 1.20 / 1.57 / 2.00
min_support=0.2, min_confidence=0.5
  Total rules: 19
  Lift min/avg/max: 1.20 / 2.65 / 5.00
min_support=0.1, min_confidence=0.3
  Total rules: 73
  Lift min/avg/max: 0.67 / 3.06 / 5.00
